# Day 11 - PCA Basics + Dimension Reduction (완성 예제)

**학습 목표**
- PCA(Principal Component Analysis) 기본 개념 이해
- explained variance ratio 분석 및 시각화
- PCA 적용 전후 분류 성능 비교

**데이터**: sklearn Digits (8x8 손글씨 숫자, 64차원)

## 1. 라이브러리 및 데이터 로드

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# 재현성 고정
tf.random.set_seed(42)
np.random.seed(42)

# Digits 데이터 로드 (8x8 = 64차원)
digits = load_digits()
X, y = digits.data, digits.target

print("=" * 50)
print("데이터 기본 정보")
print("=" * 50)
print(f"X shape      : {X.shape}")          # (1797, 64)
print(f"y shape      : {y.shape}")          # (1797,)
print(f"클래스 수    : {len(np.unique(y))}")  # 0~9
print(f"픽셀 값 범위 : {X.min()} ~ {X.max()}")

## 2. 스케일링 (PCA 전 필수)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"스케일링 후 평균 (첫 5개 특성): {X_scaled.mean(axis=0)[:5].round(4)}")
print(f"스케일링 후 표준편차 (첫 5개 특성): {X_scaled.std(axis=0)[:5].round(4)}")

## 3. PCA 적용 - 95% 분산 설명 기준으로 자동 차원 선택

`n_components=0.95` → 누적 설명 분산이 95% 이상이 되는 최소 주성분 개수를 자동 선택

In [ ]:
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("=" * 50)
print("PCA 결과 (n_components=0.95)")
print("=" * 50)
print(f"원본 차원          : {X.shape[1]}")
print(f"PCA 후 차원        : {X_pca.shape[1]}")
print(f"설명된 분산 비율 합 : {pca.explained_variance_ratio_.sum():.4f}")
print(f"\n주성분별 설명 분산 비율 (상위 10개):")
for i, ratio in enumerate(pca.explained_variance_ratio_[:10]):
    print(f"  PC{i+1:2d}: {ratio:.4f} ({ratio*100:.2f}%)")

## 4. 누적 설명 분산 시각화

In [ ]:
cumsum = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(cumsum)+1), cumsum, 'b-', linewidth=2, marker='o', markersize=4)
plt.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
plt.axvline(x=X_pca.shape[1], color='g', linestyle='--', label=f'Selected components ({X_pca.shape[1]})')
plt.xlabel('Number of Principal Components', fontsize=12)
plt.ylabel('Cumulative Explained Variance Ratio', fontsize=12)
plt.title('PCA Explained Variance Ratio', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"→ {X_pca.shape[1]}개의 주성분으로 원본 분산의 {pca.explained_variance_ratio_.sum()*100:.1f}%를 설명")

## 5. PCA 적용 전후 분류 성능 비교

동일한 MLP 구조로 **원본 64차원** vs **PCA 30차원** 성능을 비교합니다.

In [ ]:
# train/test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

### 5-1. 원본 데이터 (64차원) 학습

In [ ]:
model_orig = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
], name='Original_64dim')

model_orig.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_orig = model_orig.fit(
    X_train, y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

loss_orig, acc_orig = model_orig.evaluate(X_test, y_test, verbose=0)
print(f"[원본 64차원] Test Accuracy: {acc_orig:.4f}")

### 5-2. PCA 적용 데이터 (30차원) 학습

In [ ]:
# PCA 30차원으로 축소
pca30 = PCA(n_components=30, random_state=42)
X_train_pca = pca30.fit_transform(X_train)
X_test_pca = pca30.transform(X_test)   # 반드시 transform만 사용 (fit 금지)

print(f"PCA 후 Train shape: {X_train_pca.shape}")
print(f"PCA 설명 분산 합  : {pca30.explained_variance_ratio_.sum():.4f}")

model_pca = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(30,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
], name='PCA_30dim')

model_pca.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_pca = model_pca.fit(
    X_train_pca, y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

loss_pca, acc_pca = model_pca.evaluate(X_test_pca, y_test, verbose=0)
print(f"[PCA 30차원] Test Accuracy: {acc_pca:.4f}")

## 6. 결과 비교 및 학습 곡선

In [ ]:
print("=" * 50)
print("최종 성능 비교")
print("=" * 50)
print(f"원본 (64차원) Accuracy : {acc_orig:.4f}")
print(f"PCA  (30차원) Accuracy : {acc_pca:.4f}")
print(f"차원 축소 비율         : {30/64*100:.1f}% (약 절반)")
print(f"성능 차이              : {(acc_orig - acc_pca)*100:.2f}%p")

# 학습 곡선 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_orig.history['loss'], label='Train')
axes[0].plot(history_orig.history['val_loss'], label='Val')
axes[0].set_title('Original (64-dim) Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_pca.history['loss'], label='Train')
axes[1].plot(history_pca.history['val_loss'], label='Val')
axes[1].set_title('PCA (30-dim) Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 핵심 정리

| 항목 | 내용 |
|------|------|
| PCA 목적 | 고차원 데이터의 분산을 최대한 보존하면서 차원 축소 |
| explained_variance_ratio_ | 각 주성분이 설명하는 분산 비율 |
| n_components=0.95 | 누적 95% 분산을 설명하는 최소 주성분 개수 자동 선택 |
| 주의사항 | train에 `fit_transform`, test에는 `transform`만 사용 (데이터 누수 방지) |
| 실무 팁 | 차원이 크게 줄어도 성능이 비슷하면 학습 속도·메모리 이득을 얻음 |

**체크포인트**
- [ ] explained_variance_ratio_ 의미 이해
- [ ] 누적 분산 그래프 해석
- [ ] PCA 전후 성능 비교 완료
- [ ] fit / transform 올바른 사용